# Window Size Sensitivity Analysis
**Author:** Elsa Susana Ochoa  
**Master's Thesis — UNAM, 2018**

Analyzes how the number of negative/zero eigenvalues and the background correlation evolve over time for different window sizes T. This shows the robustness of the Power Mapping approach across different time scales.

## 1. Imports and Load Data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy import linalg as LA
from scipy import stats
import matplotlib
matplotlib.style.use('ggplot')
%matplotlib inline

with open('../data/Symbols.txt', 'r') as S:
    s = [line.split()[0] for line in S]
lensymbols = len(s)

with open('../data/Dates.txt', 'r') as D:
    d = [line.split()[0] for line in D]
lendates = len(d)

with open('../data/Gics.txt', 'r') as G:
    g = [line.split()[0] for line in G]

Prices = np.loadtxt('../data/Prices.txt')

N = 293
q = 1.05

# Year labels for x-axis (1992-2014)
year = [1992 + i for i in range(23)]

print(f'Stocks: {lensymbols}, Dates: {lendates}')

## 2. Compute Returns
Daily returns: R_t = (P_t / P_{t-1}) - 1

In [ ]:
Returns = []
for i in range(lensymbols):
    Returns.append([])
    for j in range(1, lendates):
        r = (Prices[i][j] / Prices[i][j-1]) - 1
        Returns[-1].append(r)
Returns = np.asarray(Returns)
print(f'Returns shape: {Returns.shape}')

## 3. Negative Eigenvalue Count vs Background Correlation
For each window size T, plots the number of negative eigenvalues (blue) and the mean background correlation <C_ij> (red) over time.

Key insight: as T increases relative to N, the matrix becomes less singular and Power Mapping has less work to do. The spikes in <C_ij> correspond to market stress periods (dot-com 2000, financial crisis 2008).

In [ ]:
import os
os.makedirs('../figures', exist_ok=True)

T_values = [15, 22, 30, 44, 60]

for T in T_values:
    shift = 1
    nummat = int((len(Returns[0]) - shift) / shift)
    salto = nummat / 23
    xx = np.arange(0, nummat, salto)

    # Compute background correlation and negative eigenvalue count
    CPmean = []
    negativo = []
    k = 0
    for z in range(nummat):
        mat = Returns[:, z:z+T]
        mat = np.array([(row - row.mean()) / row.std() for row in mat])
        C = np.corrcoef(mat)
        cm = (C.sum() - N) / (N**2 - N)
        CPmean.append(cm)
        Cq = np.sign(C) * (np.abs(C) ** q)
        w, _ = LA.eigh(Cq)
        negativo.append(sum(1 for v in w if v < -1e-13))

    fig, ax1 = plt.subplots(figsize=(14, 5))
    color1 = 'tab:red'
    ax1.set_ylabel(r'$\langle C_{ij}\rangle$', color=color1)
    ax1.plot(CPmean, color=color1)
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.set_xticks(xx)
    ax1.set_xticklabels(year, rotation=90)

    ax2 = ax1.twinx()
    color2 = 'tab:blue'
    ax2.set_ylabel('No. negative eigenvalues', color=color2)
    ax2.plot(negativo, color=color2)
    ax2.tick_params(axis='y', labelcolor=color2)

    plt.title(f'T={T}', fontsize=16)
    fig.tight_layout()
    plt.savefig(f'../figures/negative_eigenvalues_T{T}.png', bbox_inches='tight')
    plt.show()
    print(f'T={T} done')

## 4. Correlation Between Negative Eigenvalues and Background Correlation
Computes the rolling correlation between the number of negative eigenvalues and the background correlation <C_ij> for different window sizes T. A high correlation confirms that matrix singularity is driven by market-wide correlation spikes, not random noise.

In [ ]:
T_values = [15, 22, 30]
t = 22  # rolling window for correlation computation

fig, ax = plt.subplots(figsize=(14, 5))

for T in T_values:
    shift = 1
    nummat = int((len(Returns[0]) - shift) / shift)

    CPmean = []
    negativo = []
    for z in range(nummat):
        mat = Returns[:, z:z+T]
        mat = np.array([(row - row.mean()) / row.std() for row in mat])
        C = np.corrcoef(mat)
        CPmean.append((C.sum() - N) / (N**2 - N))
        Cq = np.sign(C) * (np.abs(C) ** q)
        w, _ = LA.eigh(Cq)
        negativo.append(sum(1 for v in w if v < -1e-13))

    # Rolling correlation
    corlist = []
    numventanas = nummat // t
    for j in range(numventanas):
        chunk_cm  = CPmean[j*t:(j+1)*t]
        chunk_neg = negativo[j*t:(j+1)*t]
        corr = np.corrcoef([chunk_cm, chunk_neg])
        corlist.append(corr[0][1])

    salto = numventanas / 23
    xx = np.arange(0, numventanas, salto)
    ax.plot(corlist, label=f'T={T}')

ax.set_xticks(xx)
ax.set_xticklabels(year, rotation=90)
ax.set_ylabel('Rolling correlation')
ax.set_title('Correlation: negative eigenvalues vs background correlation')
ax.legend()
fig.tight_layout()
plt.savefig('../figures/rolling_correlation_sensitivity.png', bbox_inches='tight')
plt.show()

## Summary
- Larger window size T reduces matrix singularity (fewer negative eigenvalues) but reduces time resolution.
- T=44 (used throughout the thesis) balances singularity control with sufficient temporal resolution to capture market dynamics.
- The number of negative eigenvalues is strongly correlated with background correlation spikes, confirming that singularity is a market phenomenon, not a statistical artifact.